In [ ]:
# # Run AdsPower in headless mode with API key and custom API port
# adspower --headless=true --api-key=45b2f684c84679a888523e55a01b563b --api-port=50325 --no-sandbox

# # Run AdsPower without sandboxing
# adspower --no-sandbox

# # Run AdsPower in headless mode with Xvfb (X virtual framebuffer) for systems without a display
# xvfb-run adspower --headless=true --api-key=45b2f684c84679a888523e55a01b563b --api-port=50325 --no-sandbox


In [ ]:
import json
import requests

url = "http://local.adspower.net:50325/api/v1/group/create"

payload = {
  "group_name": "test"
}
headers = {
  'Content-Type': 'application/json'
}

response = requests.post(url, headers=headers, json=payload)

beautified_json = json.dumps(response.json(), indent=2)

print(beautified_json)

In [ ]:
import requests

url = "http://local.adspower.net:50325/api/v1/group/list?page=1&page_size=15"

response = requests.get(url)

beautified_json = json.dumps(response.json(), indent=2)

print(beautified_json)

In [ ]:
import subprocess

def set_timezone_jakarta():
    try:
        # Command to set the timezone to Asia/Jakarta
        subprocess.run(['sudo', 'timedatectl', 'set-timezone', 'Asia/Jakarta'], check=True)
        print("Timezone successfully set to Asia/Jakarta.")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred while setting timezone: {e}")

if __name__ == "__main__":
    set_timezone_jakarta()


In [ ]:
import requests
import json

url = "http://local.adspower.net:50325/api/v1/user/list?page=1&page_size=100"

payload = {}
headers = {}

response = requests.request("GET", url, headers=headers, data=payload)

response_data = response.json()

# Extract and print only the name and user_id
for user in response_data["data"]["list"]:
    print(f"Name: {user['name']}, User ID: {user['user_id']}")


In [85]:
import json
import requests
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import sys

from profile_class import Profile
from fingerprint_config import FingerprintConfig, RandomUA
from user_proxy_config import UserProxyConfig
# this is cell is all helper functions
# Define the payload
payload = Profile(
    name='test',
    group_id='4683840',
    fingerprint_config=FingerprintConfig(
        random_ua=RandomUA(ua_system_version=['Mac OS X 13'])
        # fonts=['all']
    ),
    user_proxy_config=UserProxyConfig(proxy_soft='no_proxy')
).to_dict()

def create_profile(payload):
    url = "http://local.adspower.net:50325/api/v1/user/create"
    headers = {
        'Content-Type': 'application/json'
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response.text

def start_browser_session(user_id):
    base_url = "http://local.adspower.net:50325/api/v1"
    open_url = f"{base_url}/browser/start?user_id={user_id}"
    close_url = f"{base_url}/browser/stop?user_id={user_id}"

    resp = requests.get(open_url).json()
    print(resp)
    if resp["code"] != 0:
        print(resp["msg"])
        print("Please check user_id")
        sys.exit()

    chrome_driver = resp["data"]["webdriver"]
    service = Service(executable_path=chrome_driver)
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", resp["data"]["ws"]["selenium"])

    driver = webdriver.Chrome(service=service, options=chrome_options)
    print(driver.title)
    
    return driver, close_url

def stop_browser_session(close_url):
    resp = requests.get(close_url).json()
    if resp["code"] == 0:
        print("Browser session stopped successfully.")
    else:
        print(f"Error stopping browser session: {resp['msg']}")

def perform_signup(driver, close_url):
    def highlight(element):
        driver.execute_script("arguments[0].style.border='3px solid red'", element)

    try:
        # Open the signup page
        driver.get("https://dashboard.residential.rayobyte.com/user-area/#/signup")
        # driver.refresh()
        time.sleep(3)

        # Wait for the page to load and locate input elements
        email_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//*[@id='eml']"))
        )
        pwd_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//*[@id='pwd']"))
        )

        # Highlight and enter the email and password
        highlight(email_input)
        highlight(pwd_input)
        email_input.send_keys("crwon@edgesoftware.xyz")
        time.sleep(2)
        pwd_input.send_keys("sBpCMKDIUU4A")

        # Locate, highlight, and click the checkbox
        checkbox = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "/html/body/div[2]/div/main/div/div[2]/div/div/form/div/div[2]/div/div/div[8]/div/div/div/div/div/div/input"))
        )
        highlight(checkbox)
        checkbox.click()

        # Locate, highlight, and click the signup button
        signup_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//*[@id='app']/div/main/div/div[2]/div/div/form/div/div[2]/div/div/button/span"))
        )
        highlight(signup_btn)
        time.sleep(2)
        signup_btn.click()

        # Wait for and print the error message, if any
        try:
            error_message = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.XPATH, "//*[@id='app']/div[2]/div/div/div[2]/div/div/div/span"))
            )
            highlight(error_message)
            print("Error message content:", error_message.text)
        except Exception as e:
            print("Manual check required")
    except Exception as e:
        print("An error occurred during signup:", e)

    finally:
        # Let user decide to close the browser session
        action = input("Press 'y' to close, press any other key to keep the browser session: ")
        if action.lower() == 'y':
            requests.get(close_url)
            driver.quit()
        else:
            print("Keeping the browser session open")

def delete_all_profiles():
    base_url = "http://local.adspower.net:50325"
    list_url = f"{base_url}/api/v1/user/list?page=1&page_size=100"
    delete_url = f"{base_url}/api/v1/user/delete"
    
    # Get the list of profiles
    response = requests.get(list_url)
    if response.status_code != 200:
        print(f"Error fetching profile list: {response.status_code}")
        return
    
    profiles_data = response.json()
    profiles = profiles_data.get("data", {}).get("list", [])
    user_ids = [profile['user_id'] for profile in profiles]
    
    if not user_ids:
        print("No profiles to delete.")
        return
    
    # Delete profiles
    payload = {"user_ids": user_ids}
    delete_response = requests.post(delete_url, headers={'Content-Type': 'application/json'}, json=payload)
    
    if delete_response.status_code == 200:
        print("All profiles deleted successfully.")
    else:
        print(f"Error deleting profiles: {delete_response.status_code} - {delete_response.text}")

# Execute the function

In [80]:
# /html/body/div[2]/div/main/div/div[2]/div/div/form/div/div[2]/div/div/div[8]/div/div/div/div/div/div/input using this full xpath
elament = "/html/body/div[2]/div/main/div/div[2]/div/div/form/div/div[2]/div/div/div[8]/div/div/div/div/div/div/input"

def highlight(driver, element):
    driver.execute_script("arguments[0].style.border='3px solid red'", element)

highlight(driver, elament)

JavascriptException: Message: javascript error: Cannot set properties of undefined (setting 'border')
  (Session info: chrome=126.0.6478.57)
Stacktrace:
#0 0x55877f170312 <unknown>
#1 0x55877f16209e <unknown>
#2 0x55877ee1a587 <unknown>
#3 0x55877ee1fa8b <unknown>
#4 0x55877ee2184b <unknown>
#5 0x55877eea4863 <unknown>
#6 0x55877ee85e42 <unknown>
#7 0x55877ee57eaf <unknown>
#8 0x55877eea3c5b <unknown>
#9 0x55877ee85ae3 <unknown>
#10 0x55877ee56f31 <unknown>
#11 0x55877ee55fb9 <unknown>
#12 0x55877ee56d0c <unknown>
#13 0x55877f11a0ff <unknown>
#14 0x55877f134a75 <unknown>
#15 0x55877f1344e2 <unknown>
#16 0x55877f134f05 <unknown>
#17 0x55877f123bf7 <unknown>
#18 0x55877f1352a0 <unknown>
#19 0x55877f10c101 <unknown>
#20 0x55877f152548 <unknown>
#21 0x55877f1526e4 <unknown>
#22 0x55877f16130a <unknown>
#23 0x7f9a1b901134 <unknown>
#24 0x7f9a1b9817dc <unknown>


In [ ]:
payload = Profile(
    name='test',
    group_id='4683840',
    fingerprint_config=FingerprintConfig(
        random_ua=RandomUA(ua_system_version=['Mac OS X 13'])
        # fonts=['all']
    ),
    user_proxy_config=UserProxyConfig(proxy_soft='no_proxy')
).to_dict()

In [86]:
response_text = create_profile(payload)
# Print the response and extract the user ID
print(response.text)
user_id = json.loads(response_text)["data"]["id"]
print(f"User ID: {user_id}")
driver, close_url = start_browser_session(user_id)
perform_signup(driver, close_url)
delete_all_profiles()

{"data":{"list":[{"group_id":"4683840","group_name":"test","remark":""}],"page":1,"page_size":15},"code":0,"msg":"Success"}
User ID: kl3neyl
{'code': 0, 'msg': 'success', 'data': {'ws': {'puppeteer': 'ws://127.0.0.1:37477/devtools/browser/301e49ea-78f6-4d82-8149-8e30ce149a7f', 'selenium': '127.0.0.1:37477'}, 'debug_port': '37477', 'webdriver': '/root/.config/adspower_global/cwd_global/chrome_126/chromedriver'}}
29


In [ ]:
delete_all_profiles()